# Harmonisation and validation

Construct the canonical 1977–2025 B.9 panel, retain the 1995 overlap, and expose the four-year detailed-account gap rather than imputing it.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
pd.set_option('display.max_columns', 100)


In [2]:
from portugal_fiscal_balance.sources.banco_portugal import extract_long_series
from portugal_fiscal_balance.sources.pordata import load_balance_snapshot
from portugal_fiscal_balance.sources.cfp import extract_cfp_annual
from portugal_fiscal_balance.processing.harmonize import harmonize_balance_panel, harmonize_account_panel, build_methodology_overlap
from portugal_fiscal_balance.processing.validation import validate_balance_panel, validate_accounts
from portugal_fiscal_balance.paths import RAW
h = extract_long_series(RAW / 'banco_portugal' / 'series_longas_2023-12.xlsx')
m = load_balance_snapshot(RAW / 'pordata' / 'pordata_2785_balance_by_level_1995_2025.csv')
c = extract_cfp_annual(RAW / 'cfp' / 'cfp_sec2010_annual_general_government_2026-04-15.xlsx', RAW / 'cfp' / 'cfp_sec2010_annual_subsectors_2026-04-15.xlsx')
panel = harmonize_balance_panel(h.balances, m, c.general_government[['year','nominal_gdp_m_eur']])
accounts = harmonize_account_panel(h.accounts, c.accounts)
print(validate_balance_panel(panel))
display(build_methodology_overlap(h.balances, m))

/opt/pyvenv/lib/python3.13/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


/opt/pyvenv/lib/python3.13/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


/opt/pyvenv/lib/python3.13/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


/opt/pyvenv/lib/python3.13/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


/opt/pyvenv/lib/python3.13/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'AP S13 (%PIB)'!$C:$AK.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/opt/pyvenv/lib/python3.13/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'AP S13 (M€)'!$C:$AK.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/opt/pyvenv/lib/python3.13/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'AP S13 MTEFE (M€)'!$C:$AK.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/opt/pyvenv/lib/python3.13/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'AP S13 MTEFE OpSEC (M€)'!$C:$AK.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


/opt/pyvenv/lib/python3.13/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'AdC S1311 (%PIB)'!$C:$AF.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/opt/pyvenv/lib/python3.13/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'AdC S1311 (M€)'!$C:$AF.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/opt/pyvenv/lib/python3.13/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'ARL S1313 (%PIB)'!$C:$AF.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/opt/pyvenv/lib/python3.13/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'ARL S1313 (M€)'!$C:$AF.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/opt/pyvenv/lib/python3.13/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set 

{'n_years': 49, 'max_abs_closure_error_m_eur': 1.0, 'all_closures_within_2m_eur': True}


,metric,historical_1995_m_eur,modern_1995_m_eur,difference_m_eur
0,general_government_balance_m_eur,-4557.695302,-4611.0,-53.304698
1,central_government_balance_m_eur,-4220.739214,-4235.0,-14.260786
2,regional_local_balance_m_eur,32.579300,66.0,33.420700
3,social_security_balance_m_eur,-369.535388,-442.0,-72.464612


In [3]:
display(accounts.groupby('sector')['year'].agg(['min','max','count']))
display(validate_accounts(accounts).groupby('sector')['identity_error_m_eur'].apply(lambda x: x.abs().max()).to_frame())

,min,max,count
sector,,,
central_government,1977,2025,45
general_government,1977,2025,49
regional_local_government,1977,2025,45
social_security_funds,1977,2025,45


,identity_error_m_eur
sector,
central_government,5.456968e-12
general_government,1.136868e-12
regional_local_government,2.842171e-13
social_security_funds,9.379164e-13
